In [1]:
!git clone https://github.com/ifzhang/ByteTrack.git
%cd ByteTrack
!pip install -r requirements.txt
!python3 setup.py develop
!pip install cython
!pip install 'git+https://github.com/cocodataset/cocoapi.git#subdirectory=PythonAPI'
!pip install cython_bbox

Cloning into 'ByteTrack'...
remote: Enumerating objects: 2007, done.
remote: Counting objects: 100% (346/346), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 2007 (delta 304), reused 296 (delta 296), pack-reused 1661 (from 1)
Receiving objects: 100% (2007/2007), 79.59 MiB | 28.54 MiB/s, done.
Resolving deltas: 100% (1160/1160), done.
/content/ByteTrack
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 13.4 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: Could not find a version that satisfies the requirement onnxruntime==1.8.0 (from versions: 1.20.0, 1.20.1, 1.21.0, 1.21.1, 1.22.0, 1.22.1, 1.23.0, 1.23.1, 1.23.2, 1.24.1, 1.24.2, 1.24.3, 1.24.4, 1.25.0, 1.25.1, 1.26.0, 1.27.0, 1.28.0, 1.29.0, 1.30.0, 1.31.0, 1.31.1)
ERROR: No matching distribution found for onnxruntime==1.8.0
running develop
/usr/local/lib/pyt

In [3]:
!pip install loguru thop

  Using cached loguru-0.8.0-py3-none-any.whl.metadata (23 kB)
  Using cached thop-0.1.1.post2209072238-py3-none-any.whl.metadata (2.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [thop]


In [4]:
!unzip -q img1.zip

In [6]:
!pip install lap motmetrics

  Using cached lap-0.5.14-cp313-cp313-manylinux1_x86_64.manylinux_2_28_x86_64.manylinux_2_5_x86_64.whl.metadata (6.7 kB)
  Using cached motmetrics-1.4.0-py3-none-any.whl.metadata (20 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 12.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [motmetrics]


### Extraer detecciones de YOLOX

In [10]:
import sys
sys.path.append('.')
import os
import torch
from yolox.exp import get_exp
from tools.demo_track import Predictor
from yolox.tracking_utils.timer import Timer

# Configurar modelo con umbral bajo para capturar Dlow
exp = get_exp("exps/example/mot/yolox_x_mix_det.py", None)
exp.test_conf = 0.01  # Bajar el umbral para no descartar detecciones ocluidas
model = exp.get_model().eval().cuda().half()
ckpt = torch.load("pretrained/bytetrack_x_mot17.pth.tar", map_location="cpu")
model.load_state_dict(ckpt["model"])

predictor = Predictor(model, exp, trt_file=None, decoder=None, device="cuda", fp16=True)
timer = Timer()

img_folder = "img1" # Tu carpeta descomprimida
output_file = open("MOT17-04-yolox.txt", "w")

frames = sorted([f for f in os.listdir(img_folder) if f.endswith('.jpg')])

print("Extrayendo detecciones frame por frame...")
for frame_idx, img_name in enumerate(frames, start=1):
    img_path = os.path.join(img_folder, img_name)

    # Inferencia cruda
    outputs, img_info = predictor.inference(img_path, timer)

    if outputs[0] is not None:
        output = outputs[0].cpu().numpy()
        # Escalar coordenadas de vuelta al tamaño original de la imagen
        output[:, 0:4] /= img_info["ratio"]

        # Formatear y guardar
        for box in output:
            x1, y1, x2, y2, obj_conf, class_conf, class_pred = box
            score = obj_conf * class_conf
            w = x2 - x1
            h = y2 - y1

            # class_pred == 0 es la clase "persona" en YOLOX
            if int(class_pred) == 0:
                output_file.write(f"{frame_idx},-1,{x1:.2f},{y1:.2f},{w:.2f},{h:.2f},{score:.4f},1,-1,-1\n")

output_file.close()
print("¡Listo! Descarga el archivo MOT17-04-yolox.txt")

Extrayendo detecciones frame por frame...


/usr/local/lib/python3.13/dist-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/TensorShape.cpp:4221.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


¡Listo! Descarga el archivo MOT17-04-yolox.txt
